# FLUX LoRA Use

⚠️ **Remember to copy this notebook in your Drive and rename.**

🤗 This notebook uses [Hugging Face Diffusers](https://huggingface.co/docs/diffusers/en/index) to create pipelines for tasks such as image generation.

LoRA Adapter Weights & Scales

**This notebook shows how to**

(1) load and unload LoRA weight files from Google Drive into Diffusers text-to-image pipelines

(2) generate images with them with different LoRA scale values.

Hugging Face's docs on [loading LoRA adapters](https://huggingface.co/docs/diffusers/using-diffusers/loading_adapters#lora) explains the concepts used in this notebook.

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate huggingface_hub peft torchao


## Mount Drive

In [ ]:
# Mount your Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Hugging Face Token

In [ ]:
# Sign up at Hugging Face and create a "Read" access token (not the default "Fine-Grained" token).
# Click the 🔑 "Secrets" icon in the left sidebar.
# Enable Notebook Access, Set the Name to "HF_TOKEN", Paste your token as the Value

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

## Set Directories

In [ ]:
# Set to the Drive directory containing your LoRA weights and WEIGHTS_NAME
WEIGHTS_DIR = '/content/drive/MyDrive/iaac_genai/workflows/03_FINETUNING/02_FLUX.1/weights_FLUX.1/bof_aar_lacha'
WEIGHTS_NAME = 'bof_aar_lacha.safetensors'

In [ ]:
# Check your weights are in the right place
!ls {WEIGHTS_DIR}

## Load Pipeline

In [ ]:
from diffusers import FluxPipeline
import torch

# Create a text-to-image pipeline and load the LoRA weights
pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16, token=hf_token)
pipe.enable_model_cpu_offload()
pipe.load_lora_weights(WEIGHTS_DIR, weight_name=WEIGHTS_NAME)


# Config

In [ ]:
# Define prompt and seed, Use trigger / rare token (e.g. nnmsktch)in the prompt to trigger the LoRA
SEED = 1000
TRIGGER = "bof_aar_lacha "
PROMPT = TRIGGER + "postmodern Mediterranean apartment complex, swimming pool in foreground, Smooth, clean finishes, glass windows, concrete, painted surfaces, Cubic, angular, stepped terraces, vertical and horizontal lines, Coral red, dusty rose pink, cobalt blue, raw concrete grey, Coastal, Mediterranean, clear blue sky, ocean view, Elevated, wide shot, capturing pool and building"
print(PROMPT)

## Test LoRA at Range of Scales

In [ ]:
from IPython.display import display, Image as IPImage
import io

SCALES = [0.0, 0.25, 0.5, 0.75, 1.0]
STEPS = 20
for scale in SCALES:
    generator = torch.Generator('cuda').manual_seed(SEED)
    image = pipe(
        PROMPT,
        generator=generator,
        num_inference_steps=STEPS,
        joint_attention_kwargs={'scale': scale}
    ).images[0]
    print(f"LoRA scale: {scale}")
    display(image)

## Change Scale

In [ ]:
# Change the LoRA scale and generate the same image

SCALE = 1.0
STEPS = 40
PROMPT = TRIGGER + "Mughal street bazaar, crimson red saffron yellow ivory facades, jali screens pointed arches carved sandstone, axial reflecting pool turquoise water, diverse figures in rich layered clothing, hard midday shadow, hyper-saturated, 8K"

generator = torch.Generator('cuda').manual_seed(SEED)
image = pipe(PROMPT, generator=generator, num_inference_steps=STEPS, joint_attention_kwargs={'scale': scale}).images[0]
image

## Unload & Load New LoRA Weights

In [ ]:
# Check your weights are in the right place
!ls {WEIGHTS_DIR}

In [ ]:
# Unload the LoRA weights and load different LoRA weights *** LoRA Weights must already have been converted to ColabLoRA above

new_lora_path = '/content/drive/MyDrive/iaac_genai/workflows/03_FINETUNING/02_FLUX.1/weights_FLUX.1/bof_aar_lacha'
new_lora_weight = 'bof_aar_lacha_000001750.safetensors'

pipe.unload_lora_weights()
pipe.load_lora_weights(new_lora_path, weight_name=new_lora_weight)

In [ ]:
SCALE = 1.0
STEPS = 40
PROMPT = TRIGGER + "Mughal street bazaar, crimson red saffron yellow ivory facades, jali screens pointed arches carved sandstone, axial reflecting pool turquoise water, diverse figures in rich layered clothing, hard midday shadow, hyper-saturated, 8K"


In [ ]:
# Generate with the reloaded LoRA weights
generator = torch.Generator('cuda').manual_seed(SEED)
image = pipe(PROMPT, generator=generator, num_inference_steps=STEPS, joint_attention_kwargs={'scale': scale}).images[0]
image

## Disconnect

In [ ]:
from google.colab import runtime
runtime.unassign()